
# Импорт библиотек


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# Пути к данным


In [5]:
case_df = pd.read_parquet("../data/case_7_data_for_rd.snappy.parquet")

truth_df = pd.read_parquet("../data/truth_rd_data.snappy.parquet")

definitions_df = pd.read_excel("../data/TG_Definitions.xlsx")

In [6]:
case_df.head(10)

,rd_documentnumber,rank,rd_data,tg_ids
0,,1,"{""number"": """", ""product"": {""productName"": """", ...",[]
1,,2,"{""number"": """", ""product"": {""productName"": """", ...",[]
2,AM.01.01.01.003.R.000013.01.22,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[35]
3,AM.01.01.01.003.R.000014.01.22,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[37]
4,AM.01.01.01.003.R.000018.07.20,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[37]
5,AM.01.01.01.003.R.000021.01.23,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[37]
6,AM.01.01.01.003.R.000022.02.22,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[37]
7,AM.01.01.01.003.R.000023.02.22,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[45]
8,AM.01.01.01.003.R.000025.01.23,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....","[45, 37]"
9,AM.01.01.01.003.R.000029.01.23,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[45]


In [8]:
from pprint import pprint
pprint(case_df.loc[2, "rd_data"])

('{"type": 13, "active": true, "number": "AM.01.01.01.003.R.000013.01.22", '
 '"docNorm": "спецификация изготовителя. ", "useArea": "Для реализации '
 'населению в качестве биологически активной добавки к пище - дополнительного '
 'источника витаминов D3, В12, источника куркумина, содержащей йод. ", '
 '"nameProd": "Биологически активная добавка к пище \\"КурКумин с витаминами '
 'D3 и B12\\" (\\"CoreCumin + Vitamins D3 & B12\\"), жидкость во флаконах по '
 '5-100 мл с дозатором-капельницей или дозатором пипеткой. ", "protocol": '
 '"Взамен свидетельства о государственной регистрации '
 'AM.01.01.01.003.R.000274.10.21 от 19.10.2021 г, экспертного заключения № '
 'ЭЗ/224 от 11.10.2021 г., протокола испытаний № 1515 от 08.10.2021 г., '
 'выданных санитарно-гигиенической испытательной лабораторией ЗАО '
 '«Национальный институт здравоохранения имени академика С. Авдалбекяна» МЗ РА '
 '(аттестат аккредитации № 015/Т-081 до 14.10.2022 г.).", "firmGetName": "ООО '
 '\\"Альфа-Консалтинг\\"", 

Правило №1: tg_ids никогда не используется в качестве признака модели.(используем только для анализа)

In [9]:
type(case_df.loc[2, "tg_ids"])

numpy.ndarray

In [10]:
case_df["rank"].value_counts()

rank
1    3595803
2       7287
3        409
4         18
Name: count, dtype: int64

In [11]:
truth_df.shape #(573354, 5)
#truth_df.head()
print(truth_df.loc[2, 'rd_number'])
truth_df.iloc[0]


Условия хранения стандартные для данного вида продукции, в соответствии со статьей 3 ТР ТС 009/2011 «О безопасности парфюмерно-косметической продукции». Срок и условия хранения (годности), эксплуатации указан в прилагаемой к продукции товаросопроводительной документации и/или на упаковке и/или каждой единице продукции. Декларация выдана взамен ЕАЭС N RU Д-TR.РА02.В.66396/22


tg                                                             4
rd_type                                                      N/A
group_tnved                                                 3303
code_tnved                                            3303009000
rd_number      Декларация о соответствии распространяется на ...
Name: 0, dtype: str

In [14]:
truth_df.head()

,tg,rd_type,group_tnved,code_tnved,rd_number
0,4,N/A,3303,3303009000,Декларация о соответствии распространяется на ...
1,4,N/A,3303,3303001000,ГОСТ 31678-2012 Продукция парфюмерная жидкая. ...
2,4,N/A,3303,3303009000,Условия хранения стандартные для данного вида ...
3,4,N/A,3303,3303001000,Дата изготовления отобранных образцов (проб) п...
4,4,N/A,3303,3303001000,ГОСТ 32893-2014 Продукция парфюмерно-косметиче...


1) Большинство документов относятся к одной товарной группе, однако существует заметное количество документов с несколькими товарными группами. Возможно, задача имеет элементы multi-label классификации, хотя целевой Truth содержит только три интересующие нас группы.

In [16]:
case_df["tg_ids"].apply(len).value_counts().sort_index()

tg_ids
0         50
1    2685718
2     763343
3     132771
4      17344
5       3025
6        998
7        248
8         20
Name: count, dtype: int64

In [17]:
case_df["rank"].value_counts().sort_index()

rank
1    3595803
2       7287
3        409
4         18
Name: count, dtype: int64

In [18]:
import json

sample = json.loads(case_df.loc[2, "rd_data"])
sample.keys()

dict_keys(['type', 'active', 'number', 'docNorm', 'useArea', 'nameProd', 'protocol', 'firmGetName', 'statusGroup', 'firmMadeName', 'activeFromDate'])

In [19]:
for key in sample:
    print(f"{key}: {sample[key]}")

type: 13
active: True
number: AM.01.01.01.003.R.000013.01.22
docNorm: спецификация изготовителя. 
useArea: Для реализации населению в качестве биологически активной добавки к пище - дополнительного источника витаминов D3, В12, источника куркумина, содержащей йод. 
nameProd: Биологически активная добавка к пище "КурКумин с витаминами D3 и B12" ("CoreCumin + Vitamins D3 & B12"), жидкость во флаконах по 5-100 мл с дозатором-капельницей или дозатором пипеткой. 
protocol: Взамен свидетельства о государственной регистрации AM.01.01.01.003.R.000274.10.21 от 19.10.2021 г, экспертного заключения № ЭЗ/224 от 11.10.2021 г., протокола испытаний № 1515 от 08.10.2021 г., выданных санитарно-гигиенической испытательной лабораторией ЗАО «Национальный институт здравоохранения имени академика С. Авдалбекяна» МЗ РА (аттестат аккредитации № 015/Т-081 до 14.10.2022 г.).
firmGetName: ООО "Альфа-Консалтинг"
statusGroup: 1
firmMadeName: «Nurish.Me, Inc.»
activeFromDate: 2022-01-25


In [20]:
truth_df["tg"].value_counts()

tg
35    349165
43    223640
4        549
Name: count, dtype: int64

In [21]:
definitions = pd.read_excel("../data/TG_Definitions.xlsx", sheet_name="Определения ТГ")

definitions.shape

(164, 8)

In [22]:
definitions.info()

<class 'pandas.DataFrame'>
RangeIndex: 164 entries, 0 to 163
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Парфимерия  52 non-null     str    
 1   Unnamed: 1  30 non-null     str    
 2   Unnamed: 2  81 non-null     str    
 3   Unnamed: 3  81 non-null     str    
 4   Unnamed: 4  0 non-null      float64
 5   Unnamed: 5  0 non-null      float64
 6   Unnamed: 6  130 non-null    str    
 7   Unnamed: 7  129 non-null    str    
dtypes: float64(2), str(6)
memory usage: 54.1 KB


In [23]:
definitions.head(10)

,Парфимерия,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,Указано в ППР № 1957,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Категория в КМТ,Продукция,ТН ВЭД,Наименование ТН ВЭД,NaN,NaN,ОКПД2,Наименование ОКПД2
2,Парфюмерия,"Духи, Туалетная вода, Одеколоны",3303 00,3303 Духи и туалетная вода,NaN,NaN,20.42.11,NaN
3,Косметика,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Указано в ППР № 1681,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Косметика для губ,Косметические средства или средства для макияж...,3304100000,3304100000 Средства для макияжа губ,NaN,NaN,20.42.12.110,20.42.12.110 Средства для макияжа губ
6,Косметика для глаз,NaN,3304200000,3304200000 Средства для макияжа глаз,NaN,NaN,20.42.12.120,20.42.12.120 Средства для макияжа глаз
7,Средства для маникюра и педикюра,NaN,3304300000,3304300000 Средства для маникюра или педикюра,NaN,NaN,20.42.13.000,20.42.13.000 Средства для маникюра или педикюра
8,Тональные средства и пудра,NaN,3304910000,"3304910000 Пудра, включая компактную",NaN,NaN,20.42.14.110,20.42.14.110 Пудры и крем-пудры
9,NaN,NaN,NaN,NaN,NaN,NaN,20.42.14.130,20.42.14.130 Тальк и прочие присыпки для детей


In [24]:
dictionary = pd.read_excel(
    "../data/TG_Definitions.xlsx",
    sheet_name="Справочник значений"
)

print(dictionary.shape)
dictionary.info()
dictionary.head(10)

(1315, 4)
<class 'pandas.DataFrame'>
RangeIndex: 1315 entries, 0 to 1314
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Наименование атрибута  1241 non-null   str   
 1   Значение атрибута      1238 non-null   object
 2   Категория(-и)          1004 non-null   str   
 3   ТНВЭД                  1062 non-null   str   
dtypes: object(1), str(3)
memory usage: 171.5+ KB


,Наименование атрибута,Значение атрибута,Категория(-и),ТНВЭД
0,Парфимерия,NaN,NaN,NaN
1,Тип парфюмерии,ДУХИ,NaN,NaN
2,Тип парфюмерии,ДУШИСТАЯ ВОДА,NaN,NaN
3,Тип парфюмерии,ЛАВАНДОВАЯ ВОДА,NaN,NaN
4,Тип парфюмерии,ОДЕКОЛОН,NaN,NaN
5,Тип парфюмерии,ПАРФЮМЕРНАЯ ВОДА,NaN,NaN
6,Тип парфюмерии,СПРЕЙ ДЛЯ ТЕЛА,NaN,NaN
7,Тип парфюмерии,ТУАЛЕТНАЯ ВОДА,NaN,NaN
8,Тип парфюмерии,КОМПЛЕКТ,NaN,NaN
9,NaN,NaN,NaN,NaN


In [26]:
tnved = pd.read_excel(
    "../data/TG_Definitions.xlsx",
    sheet_name="Коды ТНВЭД по категориям"
)

print(tnved.shape)
tnved.info()
tnved.head(10)

(76, 2)
<class 'pandas.DataFrame'>
RangeIndex: 76 entries, 0 to 75
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Парфимерия  76 non-null     object
 1   Unnamed: 1  74 non-null     str   
dtypes: object(1), str(1)
memory usage: 12.5+ KB


,Парфимерия,Unnamed: 1
0,Код ТНВЭД,Наименование кода ТНВЭД
1,3303001000,3303001000 Духи
2,3303009000,3303009000 Туалетная вода
3,Косметика,NaN
4,3304100000,3304100000 Средства для макияжа губ
5,3304200000,3304200000 Средства для макияжа глаз
6,3304300000,3304300000 Средства для маникюра или педикюра
7,3304910000,"3304910000 Пудра, включая компактную"
8,3304990000,3304990000 Прочие косметические средства или с...
9,3305100000,3305100000 Шампуни


In [21]:
okpd = pd.read_excel(
    "../data/TG_Definitions.xlsx",
    sheet_name="Коды ОКПД2 по категориям"
)

print(okpd.shape)
okpd.info()
okpd.head(10)

(62, 2)
<class 'pandas.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Косметика   62 non-null     str  
 1   Unnamed: 1  62 non-null     str  
dtypes: str(2)
memory usage: 6.8 KB


,Косметика,Unnamed: 1
0,Код ОКПД2,Наименование кода ОКПД2
1,20.41.31.111,"20.41.31.111 Мыло туалетное марки ""Нейтральное"""
2,20.41.31.112,"20.41.31.112 Мыло туалетное марки ""Экстра"""
3,20.41.31.113,"20.41.31.113 Мыло туалетное марки ""Детское"""
4,20.41.31.114,"20.41.31.114 Мыло туалетное марки ""Ординарное"""
5,20.41.31.119,20.41.31.119 Мыло туалетное твердое прочее
6,20.41.31.121,20.41.31.121 Мыло хозяйственное I группы
7,20.41.31.122,20.41.31.122 Мыло хозяйственное II группы
8,20.41.31.123,20.41.31.123 Мыло хозяйственное III группы
9,20.41.31.130,20.41.31.130 Мыло туалетное жидкое


In [27]:
sample["number"] == case_df.loc[2, "rd_documentnumber"]

True

In [28]:
doc = json.loads(case_df.loc[2, "rd_data"])

doc["nameProd"]

'Биологически активная добавка к пище "КурКумин с витаминами D3 и B12" ("CoreCumin + Vitamins D3 & B12"), жидкость во флаконах по 5-100 мл с дозатором-капельницей или дозатором пипеткой. '

In [29]:
for i in range(5):
    doc = json.loads(case_df.loc[i, "rd_data"])
    print("-" * 50)
    print(doc.get("nameProd"))

--------------------------------------------------
None
--------------------------------------------------
None
--------------------------------------------------
Биологически активная добавка к пище "КурКумин с витаминами D3 и B12" ("CoreCumin + Vitamins D3 & B12"), жидкость во флаконах по 5-100 мл с дозатором-капельницей или дозатором пипеткой. 
--------------------------------------------------
Биологически активная добавка к пище «БИТ ИТ Спорт супер концентрат» («BEET IT Sport super concentrate»), жидкость в бутылках по 70-250 мл. 
--------------------------------------------------
Биологически активная добавка к пище «Глутамин Зеро» («GLUTAMINE  ZERO») со вкусами: или «Лимон», или «Голубой виноград», или «Персиковый чай со льдом», или «Арбуз».


In [30]:
sample_size = 10000

records = [
    json.loads(x)
    for x in case_df["rd_data"].head(sample_size)
]

json_df = pd.DataFrame(records)

In [31]:
json_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   number             10000 non-null  str    
 1   product            4469 non-null   object 
 2   type               9998 non-null   float64
 3   active             9996 non-null   object 
 4   docNorm            7087 non-null   str    
 5   useArea            9392 non-null   str    
 6   nameProd           9997 non-null   str    
 7   protocol           9996 non-null   str    
 8   firmGetName        9996 non-null   str    
 9   statusGroup        9996 non-null   float64
 10  firmMadeName       9997 non-null   str    
 11  activeFromDate     9996 non-null   str    
 12  dateFrom           4467 non-null   str    
 13  idStatus           4467 non-null   str    
 14  applicant          4467 non-null   object 
 15  RD_fullname        4469 non-null   str    
 16  certificate        4467 non-null  

In [34]:
json_df.isna().mean().sort_values()

number               0.0000
type                 0.0002
nameProd             0.0003
firmMadeName         0.0003
protocol             0.0004
statusGroup          0.0004
firmGetName          0.0004
active               0.0004
activeFromDate       0.0004
useArea              0.0608
docNorm              0.2913
product              0.5531
RD_fullname          0.5531
dateFrom             0.5533
idStatus             0.5533
applicant            0.5533
certificate          0.5533
declaration          0.5533
rpnStatusId          0.5533
manufacturer         0.5533
techRegulations      0.5533
certificationBody    0.5533
dtype: float64

In [35]:
json_df['type'].value_counts()

type
13.0    9998
Name: count, dtype: int64

In [36]:
json_df["statusGroup"].value_counts()

statusGroup
1.0    9581
2.0     415
Name: count, dtype: int64

In [37]:
json_df["nameProd"].str.len().describe()

count    9997.000000
mean      119.515355
std        75.724639
min        11.000000
25%        64.000000
50%       106.000000
75%       158.000000
max      2507.000000
Name: nameProd, dtype: float64

In [38]:
json_df["docNorm"].str.len().describe()

count    7087.000000
mean       33.659659
std        32.361813
min         0.000000
25%        15.000000
50%        29.000000
75%        44.000000
max       383.000000
Name: docNorm, dtype: float64

In [39]:
json_df["nameProd"].sample(10, random_state=42)

6252                         Ароматизатор «285744 Малины»
4684    Комплексная пищевая добавка КОМБИ СЕРВЕЛАТ ТРА...
1731    Специализированный пищевой продукт диетическог...
4742    Комплексная пищевая добавка МОЛОЧНАЯ ЭКСТРА, а...
4521    Краска стирол-акриловая водно-дисперсионная ВД...
6340             Ароматизатор пищевой «Клубника ELL 1864»
576     Биологически активная добавка к пище: «Комплек...
5202    Средство для для облегчения глажения изделий и...
6363             Натуральный ароматизатор «912166 Кетчуп»
439     Биологически активная добавка к пище «Железо х...
Name: nameProd, dtype: str

In [40]:
json_df["docNorm"].sample(10, random_state=42)

6252                                                  NaN
4684                                                     
1731                          спецификация производителя.
4742                                                     
4521    ТУ BY 200026232.001-2010, РЦ РБ 200026232.001-...
6340                              ТУ 9145-001-66969518-11
576                         ТУ 10.89.19-012-28920794-2024
5202    ТУ BY 200574861.018-2010, РЦ BY 200574861.104-...
6363                                                     
439                         ТУ 10.89.19-125-16013430-2021
Name: docNorm, dtype: str

In [41]:
json_df["useArea"].sample(10, random_state=42)

6252    при производстве чайного листа (пакетированног...
4684    производство пищевых продуктов (мясные, в т.ч....
1731    в качестве специализированного пищевого продук...
4742    производство пищевых продуктов (мясные, в т.ч....
4521                                                  NaN
6340                   при производстве пищевых продуктов
576     Условия реализации: места реализации определяю...
5202                                                     
6363    в пищевой промышленности в дозировке 4% при пр...
439     качестве БАД к пище - дополнительного источник...
Name: useArea, dtype: str

In [ ]:
idx1 = json_df["product"].first_valid_index()
idx1

0

In [ ]:
sample = json.loads(case_df.loc[idx1, "rd_data"])
type(sample["product"])

dict

In [49]:
pprint(sample["product"])

{'identification': [], 'productName': ''}


In [50]:
idx2 = json_df["manufacturer"].first_valid_index()
idx2

3

In [53]:
sample2 = json.loads(case_df.loc[idx2, "rd_data"])
pprint(sample2["manufacturer"])

{'GLN': '',
 'address': '',
 'filialAddresses': '',
 'inn': '',
 'name': '',
 'type': ''}


In [54]:
idx3 = json_df["certificate"].first_valid_index()
idx3

3

In [55]:
sample3 = json.loads(case_df.loc[idx3, "rd_data"])
pprint(sample3["certificate"])

{'certEndDate': '',
 'certRegDate': '',
 'filialAddresses': '',
 'idCertScheme': '',
 'idCertType': '',
 'issueBasis': '',
 'number': ''}


In [57]:
idx4 = json_df["declaration"].first_valid_index()
idx4
sample4 = json.loads(case_df.loc[idx4, "rd_data"])
pprint(sample4["declaration"])

{'declEndDate': '',
 'declRegDate': '',
 'filialAddresses': '',
 'idDeclScheme': '',
 'idDeclType': '',
 'number': ''}


In [59]:
sample["product"].keys()

dict_keys(['productName', 'identification'])

In [60]:
case_df["rd_documentnumber"].duplicated().sum()

np.int64(7714)

In [61]:
duplicates = (
    case_df
    .groupby("rd_documentnumber")["rank"]
    .agg(["count", "min", "max"])
)

duplicates.sort_values("count", ascending=False).head(20)

,count,min,max
rd_documentnumber,,,
ЕАЭС N RU Д-RU.РА01.В.23496/21,4,1,4
ЕАЭС N RU Д-RU.РА01.В.26974/21,4,1,4
ЕАЭС N RU Д-RU.РА01.В.35506/21,4,1,4
ЕАЭС N RU Д-RU.РА01.В.02855/21,4,1,4
ЕАЭС N RU Д-RU.РА01.В.23568/21,4,1,4
ЕАЭС N RU Д-RU.РА01.В.24237/21,4,1,4
ЕАЭС N RU Д-RU.РА01.В.23341/21,4,1,4
ЕАЭС N RU Д-RU.РА01.В.28429/21,4,1,4
ЕАЭС N RU Д-RU.РА01.В.37034/21,4,1,4


In [62]:
truth_df["rd_number"].sample(20, random_state=42).tolist()

['ЕАЭС N RU Д-KR.РА01.В.87925/23',
 'ЕАЭС N RU Д-RU.РА02.В.14528/23',
 'ЕАЭС N RU Д-RU.РА07.В.80755/22',
 'ЕАЭС N RU Д-RU.РА02.В.95459/21',
 'ЕАЭС N RU Д-RU.РА09.В.53146/24',
 'ЕАЭС N RU Д-RU.РА04.В.11412/25',
 'ЕАЭС N RU Д-DE.РА06.В.27005/23',
 'ЕАЭС N RU Д-RU.РА10.В.77082/23',
 'ЕАЭС N RU Д-RU.РА05.В.26694/24',
 'RU.30.АЦ.02.015.Е.000170.04.24',
 'RU.54.НС.01.015.Е.000985.11.23',
 'ЕАЭС N RU Д-RU.РА02.В.45118/23',
 'ЕАЭС N RU Д-RU.РА01.В.41529/25',
 'ЕАЭС N RU Д-RU.РА07.В.39083/23',
 'ЕАЭС № BY/112 11.01. ТР009 011 06175',
 'ЕАЭС N RU Д-RU.РА02.В.83266/24',
 'ЕАЭС N RU Д-BE.РА10.В.58348/23 ||| ЕАЭС N RU Д-BE.РА11.В.25898/25',
 'ЕАЭС N RU Д-DE.РА01.В.60876/25',
 'KG.11.01.09.001.R.005950.11.24',
 'ЕАЭС N RU Д-RU.РА07.В.16997/25']

In [63]:
import random

sample_numbers = case_df["rd_documentnumber"].dropna().sample(10, random_state=42)

sample_numbers.tolist()

['ЕАЭС N RU Д-RU.РА04.В.67842/23',
 'ЕАЭС AM-020/S.B-0044-2024',
 'ЕАЭС N RU Д-PL.СП28.В.10672',
 'ЕАЭС № BY/112 11.02. ТР017 122 00265',
 'RU.77.99.32.009.Е.003170.02.15',
 'ТС RU С-TR.АУ04.В.04607',
 'ЕАЭС N RU Д-RU.РА06.А.44458/25',
 'ЕАЭС N RU Д-RU.АЖ24.В.00857',
 'ЕАЭС RU С-TJ.НЕ16.В.00405/21',
 'ЕАЭС N RU Д-RU.РА10.В.20651/25']

In [64]:
for number in sample_numbers:
    matches = truth_df["rd_number"].str.contains(
        number,
        regex=False,
        na=False
    ).sum()
    
    print(number, '->', matches)

ЕАЭС N RU Д-RU.РА04.В.67842/23 -> 0
ЕАЭС AM-020/S.B-0044-2024 -> 0
ЕАЭС N RU Д-PL.СП28.В.10672 -> 0
ЕАЭС № BY/112 11.02. ТР017 122 00265 -> 0
RU.77.99.32.009.Е.003170.02.15 -> 0
ТС RU С-TR.АУ04.В.04607 -> 0
ЕАЭС N RU Д-RU.РА06.А.44458/25 -> 0
ЕАЭС N RU Д-RU.АЖ24.В.00857 -> 0
ЕАЭС RU С-TJ.НЕ16.В.00405/21 -> 0
ЕАЭС N RU Д-RU.РА10.В.20651/25 -> 0


In [ ]:
for i, row in enumerate(case_df["rd_data"].head(50000)):
    data = json.loads(row)

    product = data.get("product")

    if isinstance(product, dict):
        ident = product.get("identification")

        if ident:
            print(i)
            print(ident)
            break

In [67]:
from collections import Counter
import json

counter = Counter()

for row in case_df["rd_data"].head(10000):
    data = json.loads(row)
    counter[tuple(sorted(data.keys()))] += 1

counter.most_common(10)

[(('RD_fullname',
   'active',
   'activeFromDate',
   'applicant',
   'certificate',
   'certificationBody',
   'dateFrom',
   'declaration',
   'docNorm',
   'firmGetName',
   'firmMadeName',
   'idStatus',
   'manufacturer',
   'nameProd',
   'number',
   'product',
   'protocol',
   'rpnStatusId',
   'statusGroup',
   'techRegulations',
   'type',
   'useArea'),
  4467),
 (('active',
   'activeFromDate',
   'firmGetName',
   'firmMadeName',
   'nameProd',
   'number',
   'protocol',
   'statusGroup',
   'type',
   'useArea'),
  2642),
 (('active',
   'activeFromDate',
   'docNorm',
   'firmGetName',
   'firmMadeName',
   'nameProd',
   'number',
   'protocol',
   'statusGroup',
   'type',
   'useArea'),
  2282),
 (('active',
   'activeFromDate',
   'docNorm',
   'firmGetName',
   'firmMadeName',
   'nameProd',
   'number',
   'protocol',
   'statusGroup',
   'type'),
  337),
 (('active',
   'activeFromDate',
   'firmGetName',
   'firmMadeName',
   'nameProd',
   'number',
   'proto

In [70]:
json_df["product"].dropna().head(10)

0             {'productName': '', 'identification': []}
1             {'productName': '', 'identification': []}
3     {'tnved': '', 'productInfo': '', 'productName'...
4     {'tnved': '', 'productInfo': '', 'productName'...
5     {'tnved': '', 'productInfo': '', 'productName'...
6     {'tnved': '', 'productInfo': '', 'productName'...
7     {'tnved': '', 'productInfo': '', 'productName'...
8     {'tnved': '', 'productInfo': '', 'productName'...
9     {'tnved': '', 'productInfo': '', 'productName'...
10    {'tnved': '', 'productInfo': '', 'productName'...
Name: product, dtype: object

In [71]:
from collections import Counter
import json

product_keys = Counter()

for row in case_df["rd_data"].head(10000):
    data = json.loads(row)

    product = data.get("product")

    if isinstance(product, dict):
        product_keys[tuple(sorted(product.keys()))] += 1

product_keys.most_common(10)

[(('idObjectType',
   'idProductOrigin',
   'identification',
   'productInfo',
   'productName',
   'tnved'),
  4467),
 (('identification', 'productName'), 2)]

In [79]:
import json

for row in case_df["rd_data"]:
    data = json.loads(row)

    product = data.get("product")

    if isinstance(product, dict):
        print(product.keys())
        break

dict_keys(['productName', 'identification'])


In [76]:
from collections import Counter

object_types = Counter()

for row in case_df["rd_data"]:
    data = json.loads(row)

    product = data.get("product")

    if isinstance(product, dict):
        object_types[product.get("idObjectType")] += 1

object_types.most_common()

[('Серийный выпуск', 2232563),
 ('Партия', 594064),
 ('Единичное изделие', 437529),
 ('', 121295),
 (None, 139)]

In [77]:
object_types = Counter()

for row in case_df["rd_data"]:
    data = json.loads(row)

    product = data.get("product")

    if isinstance(product, dict):
        object_types[product.get("idProductOrigin")] += 1

object_types.most_common()

[('РОССИЯ', 1170029),
 ('', 937743),
 (None, 309802),
 ('КИТАЙ', 252255),
 ('ИТАЛИЯ', 95426),
 ('ГЕРМАНИЯ', 86228),
 ('ФРАНЦИЯ', 63954),
 ('БЕЛАРУСЬ', 63344),
 ('ТУРЦИЯ', 45099),
 ('КОРЕЯ, РЕСПУБЛИКА', 36560),
 ('СОЕДИНЕННЫЕ ШТАТЫ', 30539),
 ('ЯПОНИЯ', 22116),
 ('ИСПАНИЯ', 21828),
 ('СОЕДИНЕННОЕ КОРОЛЕВСТВО', 20414),
 ('ПОЛЬША', 17086),
 ('УЗБЕКИСТАН', 14538),
 ('НИДЕРЛАНДЫ', 12466),
 ('ШВЕЙЦАРИЯ', 12219),
 ('ИНДИЯ', 11562),
 ('ВЬЕТНАМ', 9260),
 ('ГОНКОНГ', 8474),
 ('ТАЙВАНЬ (КИТАЙ)', 8395),
 ('ФИНЛЯНДИЯ', 8057),
 ('БЕЛЬГИЯ', 7584),
 ('ГРЕЦИЯ', 7188),
 ('ИРАН, ИСЛАМСКАЯ РЕСПУБЛИКА', 7068),
 ('УКРАИНА', 5630),
 ('ШВЕЦИЯ', 5533),
 ('ЧЕХИЯ', 5041),
 ('ТАИЛАНД', 4428),
 ('АВСТРИЯ', 4194),
 ('ИЗРАИЛЬ', 4158),
 ('ДАНИЯ', 3787),
 ('СЛОВЕНИЯ', 3758),
 ('БАНГЛАДЕШ', 3735),
 ('КИРГИЗИЯ', 3682),
 ('КАЗАХСТАН', 3449),
 ('КАНАДА', 3362),
 ('ИНДОНЕЗИЯ', 2880),
 ('ИРЛАНДИЯ', 2756),
 ('СЕРБИЯ', 2496),
 ('БОЛГАРИЯ', 2479),
 ('ОБЪЕДИНЕННЫЕ АРАБСКИЕ ЭМИРАТЫ', 2232),
 ('МАЛАЙЗИЯ', 2146),
 ('МЕКСИКА', 1909

In [81]:
import json

best = None
best_len = 0

for row in case_df["rd_data"]:
    data = json.loads(row)

    product = data.get("product")

    if isinstance(product, dict):
        if len(product) > best_len:
            best_len = len(product)
            best = product

print(best_len)
print(best.keys())

for k, v in best.items():
    print("=" * 40)
    print(k)
    print(type(v))
    print(v)

6
dict_keys(['tnved', 'productInfo', 'productName', 'idObjectType', 'identification', 'idProductOrigin'])
tnved
<class 'str'>

productInfo
<class 'str'>

productName
<class 'str'>

idObjectType
<class 'str'>

identification
<class 'list'>
[]
idProductOrigin
<class 'str'>



In [83]:
import json

max_keys = 0
best = None

for row in case_df["rd_data"]:
    data = json.loads(row)

    for key, value in data.items():
        if isinstance(value, dict):
            if len(value) > max_keys:
                max_keys = len(value)
                best = value

print(max_keys)
print(best.keys())

8
dict_keys(['inn', 'ogrn', 'type', 'email', 'phone', 'address', 'fullName', 'directorName'])


In [91]:
perfume_keywords = ["духи", 
                    "парфюмерная вода", 
                    "туалетная вода", 
                    "одеколон",
                    "parfum",
                    "perfume"]

cosmetic_keywords = [
    "крем",
    "шампун",
    "лосьон",
    "бальзам",
    "маска"
]

oil_keywords = [
    "моторное масло",
    "масло моторное",
    "5w",
    "10w",
    "sae",
    "api"
]

In [85]:
def count_keyword(text_series, keywords):
    text = text_series.fillna("").str.lower()
    
    result = {}

    for word in keywords:
        count = text.str.contains(word, regex=False).sum()
        result[word] = count
        
    return (
        pd.Series(result)
        .sort_values(ascending=False)
    )

In [86]:
print("===Парфюмерия===")
display(count_keyword(json_df["nameProd"], perfume_keywords))

print("===Косметика===")
display(count_keyword(json_df["nameProd"], cosmetic_keywords))

print("===Моторные масла===")
display(count_keyword(json_df["nameProd"], oil_keywords))

===Парфюмерия===


туалетная вода      1
духи                0
парфюмерная вода    0
одеколон            0
dtype: int64

===Косметика===


крем       384
шампун     111
бальзам     81
лосьон      79
маска       39
dtype: int64

===Моторные масла===


api               11
5w                 1
масло моторное     0
моторное масло     0
10w                0
sae                0
dtype: int64

In [87]:
json_df.loc[
    json_df["nameProd"]
        .fillna("")
        .str.lower()
        .str.contains("духи"),
    "nameProd"
].head(10)

Series([], Name: nameProd, dtype: str)

7      Биологически активная добавка к пище BioTechUS...
30     Биологически активная добавка к пище BioTechUS...
65     Биологически активная добавка к пище Genius Nu...
167    Биологически активная добавка к пище «ШЕДОВЭЙ»...
168    Биологически активная добавка к пище «ШЕДОВЭЙ ...
221    Биологически активная добавка к пище Scitec Nu...
322                      Печенье затяжное  «Кремульки». 
339    Вода минеральная природная лечебно-столовая пи...
340    Вода минеральная природная питьевая лечебно-ст...
342    Вода минеральная природная лечебно-столовая кр...
Name: nameProd, dtype: str

In [89]:
sample_size = 10000

sample_case = (
    case_df
    .sample(sample_size, random_state=42)
    .reset_index(drop=True)
)

records = [
    json.loads(x)
    for x in sample_case["rd_data"]
]

json_df = pd.DataFrame(records)

In [92]:
print("=== Парфюмерия ===")
display(count_keyword(json_df["nameProd"], perfume_keywords))

print("=== Косметика ===")
display(count_keyword(json_df["nameProd"], cosmetic_keywords))

print("=== Моторные масла ===")
display(count_keyword(json_df["nameProd"], oil_keywords))

=== Парфюмерия ===


perfume             3
parfum              2
одеколон            1
духи                0
парфюмерная вода    0
туалетная вода      0
dtype: int64

=== Косметика ===


крем       89
шампун     25
лосьон     21
маска      18
бальзам    13
dtype: int64

=== Моторные масла ===


api               2
моторное масло    0
масло моторное    0
5w                0
10w               0
sae               0
dtype: int64

In [94]:
for tg in [4, 35, 43]:
    print("=" * 80)
    print("TG =", tg)
    print()

    display(
        truth_df.loc[
            truth_df["tg"] == str(tg),
            "rd_number"
        ].head(10)
    )

TG = 4



0    Декларация о соответствии распространяется на ...
1    ГОСТ 31678-2012 Продукция парфюмерная жидкая. ...
2    Условия хранения стандартные для данного вида ...
3    Дата изготовления отобранных образцов (проб) п...
4    ГОСТ 32893-2014 Продукция парфюмерно-косметиче...
5    "ГОСТ 31678-2012 ""Продукция парфюмерная жидка...
6    Декларация соответствия распространяется на пр...
7    Green Almond & Redcurrant, торговой марки Jo M...
8      - Парфюмерная вода для мужчин DAVID BECKHAM ...
9    Декларация соответствия распространяется на пр...
Name: rd_number, dtype: str

TG = 35



224189    ЕАЭС N RU Д-RU.РА02.В.80355/22
224190    RU.77.01.34.001.Е.002952.12.16
224191    KG.11.01.09.001.R.006012.10.22
224192    ЕАЭС N RU Д-RU.ПК08.В.01644/20
224193    ЕАЭС N RU Д-RU.ПК08.В.01644/20
224194    ЕАЭС N RU Д-RU.ПК08.В.01644/20
224195    ЕАЭС N RU Д-RU.ПК08.В.01644/20
224196    ЕАЭС N RU Д-RU.ПК08.В.01644/20
224197    ЕАЭС N RU Д-RU.ПК08.В.01644/20
224198    ЕАЭС N RU Д-RU.ПК08.В.01644/20
Name: rd_number, dtype: str

TG = 43



549    ЕАЭС N RU Д-FR.РА04.В.75724/22
550    ЕАЭС N RU Д-RU.РА09.В.31300/22
551    ЕАЭС N RU Д-RU.РА05.В.60070/22
552    ЕАЭС N RU Д-RU.РА05.В.10364/23
553    ЕАЭС N RU Д-RU.РА05.В.10364/23
554    ЕАЭС N RU Д-RU.РА05.В.10364/23
555    ЕАЭС N RU Д-RU.РА05.В.10364/23
556    ЕАЭС N RU Д-RU.РА05.В.10364/23
557    ЕАЭС N RU Д-RU.РА05.В.10364/23
558    ЕАЭС N RU Д-RU.РА05.В.10364/23
Name: rd_number, dtype: str

In [95]:
truth_df.groupby("tg")["rd_type"].value_counts()

tg  rd_type
35  ДС         259223
    СГР         86947
    СС           2995
4   N/A           549
43  ДС         221884
    СГР          1026
    СС            730
Name: count, dtype: int64

In [96]:
truth_df.loc[
    truth_df["tg"] == "4",
    "rd_number"
].sample(20, random_state=42)

195    Декларация о соответствии распространяется на ...
79      - Парфюмерная вода для женщин Calvin Klein Et...
479    "п.п. 3.1.1, 3.1.5, 3.1.6; 3.2; 3.3.1; 3.4.3, ...
109    Продукция парфюмерно-косметическая жидкая: оде...
473                                          ""FEDERAL""
490    Декларация о соответствии распространяется на ...
84     Договор поставки № KGD-1728 от 27.01.2023 года...
368    Условия хранения стандартные для данного вида ...
132    Декларация о соответствии распространяется на ...
364                                           20.06.2023
184    Декларация соответствия распространяется на пр...
10     Kirke, торговой марки Tiziana Terenzi  Cruz de...
73     Декларация о соответствии распространяется на ...
220                                              SAFANAD
278    "Декларация о соответствии распространяется на...
82     "ГОСТ 31678-2012 ""Продукция парфюмерная жидка...
382    Декларация соответствия распространяется на пр...
6      Декларация соответствия 

In [97]:
truth_df.groupby("tg")["code_tnved"].nunique()

tg
35    74
4      2
43    32
Name: code_tnved, dtype: int64

In [98]:
for tg in ["4", "35", "43"]:
    print("=" * 80)
    print("TG =", tg)

    print(
        truth_df.loc[
            truth_df["tg"] == tg,
            "code_tnved"
        ]
        .value_counts()
        .head(15)
    )

TG = 4
code_tnved
3303009000    335
3303001000    161
Name: count, dtype: int64
TG = 35
code_tnved
3304990000    82940
3305900009    47024
3402500000    43973
3304300000    35844
3401300000    26808
3305100000    18077
3307490000    16091
3304100000    11343
3307300000    11002
3401110001    10350
3304200000     7677
3401209000     7271
3808948000     5497
3307200000     4496
3307900008     2907
Name: count, dtype: int64
TG = 43
code_tnved
2710198200           97609
2710198800           37186
3403199000           32767
3820000000           25145
3403990000           19389
3403191000            7653
3819000000            3808
2710                    18
 VW 502.00/505.00       11
 VW 501.00/505.00        8
3403                     7
 MB 229.51/"""           7
 Renault 0700            5
 ACEA A3                 3
 PO"""                   3
Name: count, dtype: int64


In [100]:
new_truth = pd.read_parquet("../data/2_truth_rd_data.snappy.parquet")

In [101]:
new_truth

,tg,rd_type,group_tnved,code_tnved,rd_number,rd_date
0,4,N/A,3303,3303009000,ЕАЭС N RU Д-IT.РА03.В.08011/25,2025-03-25
1,4,N/A,3303,3303009000,ЕАЭС N RU Д-FR.РА01.В.63599/25,2025-02-03
2,4,N/A,3303,3303009000,ЕАЭС N RU Д-FR.РА08.В.83499/24,2024-09-27
3,4,N/A,3303,3303009000,ЕАЭС N RU Д-ES.РА02.В.52976/25,2025-02-28
4,4,N/A,3303,3303009000,ЕАЭС N RU Д-ES.РА09.В.91241/23,2023-11-22
...,...,...,...,...,...,...
577255,35,СГР,3307,3307490000,RU.08.08.09.015.Е.000383.10.25,NaN
577256,35,СГР,3307,3307490000,RU.08.08.09.015.Е.000383.10.25,NaN
577257,35,СГР,3307,3307490000,RU.08.08.09.015.Е.000383.10.25,NaN
577258,35,СГР,3307,3307490000,RU.08.08.09.015.Е.000383.10.25,NaN


In [102]:
sample_truth = new_truth["rd_number"].sample(100, random_state=42)

matches = []

for number in sample_truth:
    if (case_df["rd_documentnumber"] == number).any():
        matches.append(number)
        
print(len(matches))        

94


In [103]:
new_truth["rd_date"].isna().mean()

np.float64(0.6066936908845234)

In [104]:
new_truth.groupby("tg")["rd_date"].apply(lambda x: x.notna().mean())

tg
35    0.000000
4     0.999776
43    0.995287
Name: rd_date, dtype: float64

In [105]:
matches = []

case_lookup = (
    case_df
    .set_index("rd_documentnumber")
)

for number in sample_truth:
    if number in case_lookup.index:
        matches.append(number)

matches[:5]

['ЕАЭС N RU Д-RU.РА01.В.52411/21',
 'ВП RU Д-AE.РА01.А.70053/25',
 'ЕАЭС N RU Д-CN.РА03.В.78441/25',
 'ЕАЭС N RU Д-KR.РА01.В.89075/23',
 'RU.30.АЦ.02.015.Е.000307.07.25']

In [106]:
number = matches[0]

case_df.loc[
    case_df["rd_documentnumber"] == number,
    ["rd_documentnumber", "rank"]
]

,rd_documentnumber,rank
2043160,ЕАЭС N RU Д-RU.РА01.В.52411/21,1
2043161,ЕАЭС N RU Д-RU.РА01.В.52411/21,2


In [107]:
new_truth.loc[
    new_truth["rd_number"] == number
]

,tg,rd_type,group_tnved,code_tnved,rd_number,rd_date
351494,35,ДС,3401,3401300000,ЕАЭС N RU Д-RU.РА01.В.52411/21,NaN
351495,35,ДС,3401,3401300000,ЕАЭС N RU Д-RU.РА01.В.52411/21,NaN
351496,35,ДС,3401,3401300000,ЕАЭС N RU Д-RU.РА01.В.52411/21,NaN
351526,35,ДС,3401,3401300000,ЕАЭС N RU Д-RU.РА01.В.52411/21,NaN
351527,35,ДС,3401,3401300000,ЕАЭС N RU Д-RU.РА01.В.52411/21,NaN
351528,35,ДС,3401,3401300000,ЕАЭС N RU Д-RU.РА01.В.52411/21,NaN
